"""
Dataset Creation using Florence-2 and Local Flickr30k Dataset
This script creates a training dataset for object detection using Florence-2 model
to generate annotations for person and pet (dog, cat, horse) classes.
"""

In [2]:
# %pip install torch
# %pip install einops timm
# %pip install albumentations opencv-python


### Florence Model Compatibility

To successfully load the Florence-2 model, we needed to downgrade the `transformers` library to version **4.51.1**.


In [3]:
%pip install transformers==4.51.1 -U

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: C:\Users\Kero\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import os
import json
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM
from tqdm import tqdm
import numpy as np
from collections import defaultdict
import shutil
import random


C:\Users\Kero\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import torch
print(torch.cuda.is_available())    # should be True
print(torch.version.cuda)           # should show your CUDA version

True
12.1


In [17]:
# Configuration
class Config:
    # Florence-2 settings
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    # Dataset settings
    MIN_OBJECT_SIZE = 32  # Minimum object size in pixels
    MAX_OBJECT_SIZE = 640  # Maximum object size in pixels
    MIN_IMAGE_SIZE = 224  # Minimum image dimension
    MAX_IMAGE_SIZE = 1024  # Maximum image dimension
    
    # Classes
    TARGET_CLASSES = ['person', 'pet']
    PET_CLASSES = ['dog', 'cat', 'horse']
    
    # Dataset requirements
    TRAIN_IMAGES_PER_CLASS = 700  
    VAL_IMAGES_PER_CLASS = 102
    
    # Local Flickr30k path
    FLICKR30K_PATH = "flickr30k_images"  # Path to local Flickr30k folder
    
    # Output directories
    OUTPUT_DIR = "object_detection_dataset"
    TRAIN_DIR = os.path.join(OUTPUT_DIR, "train")
    VAL_DIR = os.path.join(OUTPUT_DIR, "val")
    ANNOTATIONS_DIR = os.path.join(OUTPUT_DIR, "annotations")

In [7]:
class DatasetCreatorBase:
    def __init__(self, config):
        self.config = config
        self.pet_type_stats = {
            'dog': {'train': 0, 'val': 0},
            'cat': {'train': 0, 'val': 0},
            'horse': {'train': 0, 'val': 0}
        }


class DatasetUtilities(DatasetCreatorBase):
    def get_pet_type(self, label):
        for pet_type in self.config.PET_CLASSES:
            if pet_type in label:
                return pet_type
        return None

    def resize_image_if_needed(self, image):
        w, h = image.size
        if (w < self.config.MIN_IMAGE_SIZE or h < self.config.MIN_IMAGE_SIZE or
            w > self.config.MAX_IMAGE_SIZE or h > self.config.MAX_IMAGE_SIZE):
            if w < self.config.MIN_IMAGE_SIZE or h < self.config.MIN_IMAGE_SIZE:
                scale = max(self.config.MIN_IMAGE_SIZE / w, self.config.MIN_IMAGE_SIZE / h)
            else:
                scale = min(self.config.MAX_IMAGE_SIZE / w, self.config.MAX_IMAGE_SIZE / h)
            image = image.resize((int(w * scale), int(h * scale)), Image.Resampling.LANCZOS)
        return image


In [8]:
class FlorenceModelHandler(DatasetUtilities):
    def setup_directories(self):
        for dir_path in [self.config.TRAIN_DIR, self.config.VAL_DIR, self.config.ANNOTATIONS_DIR]:
            os.makedirs(dir_path, exist_ok=True)
            os.makedirs(os.path.join(dir_path, "images"), exist_ok=True)

    def load_florence_model(self):
        print("Loading Florence-2 model...")
        model_id = 'microsoft/Florence-2-large'
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True).to(self.device)
        self.model.eval()
        print(f"Model loaded successfully on {self.device}!")

    def load_local_images(self):
        print(f"Loading images from {self.config.FLICKR30K_PATH}...")
        if not os.path.exists(self.config.FLICKR30K_PATH):
            raise ValueError(f"Flickr30k folder not found at {self.config.FLICKR30K_PATH}")
        image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.JPEG', '.JPG', '.PNG', '.BMP'}
        self.image_paths = [
            os.path.join(self.config.FLICKR30K_PATH, file)
            for file in os.listdir(self.config.FLICKR30K_PATH)
            if any(file.endswith(ext) for ext in image_extensions)
        ]
        print(f"Found {len(self.image_paths)} images.")
        random.shuffle(self.image_paths)

    def run_florence_detection(self, image):
        if image.mode != 'RGB':
            image = image.convert('RGB')
        inputs = self.processor(text="<OD>", images=image, return_tensors="pt").to(self.device)
        with torch.no_grad():
            generated_ids = self.model.generate(
                input_ids=inputs["input_ids"],
                pixel_values=inputs["pixel_values"],
                max_new_tokens=1024,
                num_beams=3
            )
        decoded = self.processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
        return self.processor.post_process_generation(decoded, task="<OD>", image_size=(image.width, image.height))


In [20]:
class AnnotationHandler(FlorenceModelHandler):
    def filter_and_convert_annotations(self, florence_output, image_width, image_height):
        if '<OD>' not in florence_output:
            return []
        detections = florence_output['<OD>']
        if 'bboxes' not in detections or 'labels' not in detections:
            return []

        # Per-class thresholds (adjust as needed)
        THRESHOLDS = {
            'person': 0.57,   # stricter
            'dog': 0.67,      # stricter
            'cat': 0.10,      # relaxed
            'horse': 0.2     # relaxed
        }

        scored_annotations = []

        for bbox, label in zip(detections['bboxes'], detections['labels']):
            label_lower = label.lower()
            class_name = None
            pet_type = None

            # Class identification
            if 'person' in label_lower or 'people' in label_lower or 'human' in label_lower:
                class_name = 'person'
                threshold = THRESHOLDS['person']
            else:
                for pet in self.config.PET_CLASSES:
                    if pet in label_lower:
                        class_name = 'pet'
                        pet_type = pet
                        threshold = THRESHOLDS.get(pet, 0.1)
                        break

            if class_name is None:
                continue  # Not a relevant class

            x1, y1, x2, y2 = bbox
            width, height = x2 - x1, y2 - y1
            area = width * height

            if not (self.config.MIN_OBJECT_SIZE <= width <= self.config.MAX_OBJECT_SIZE and
                    self.config.MIN_OBJECT_SIZE <= height <= self.config.MAX_OBJECT_SIZE):
                continue

            # Scoring
            cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
            center_dx = abs(cx - image_width / 2) / image_width
            center_dy = abs(cy - image_height / 2) / image_height
            center_score = 1.0 - (center_dx + center_dy) / 2
            area_score = area / (image_width * image_height)
            score = 0.6 * area_score + 0.4 * center_score

            if score < threshold:
                continue  # Too weak for this class

            scored_annotations.append({
                'class': class_name,
                'bbox': [x1, y1, x2, y2],
                'area': area,
                'original_label': label,
                'score': score,
                'pet_type': pet_type  
            })

        return sorted(scored_annotations, key=lambda a: a['score'], reverse=True)[:3]



    def create_coco_format_annotation(self, image_id, image_filename, width, height, annotations):
        coco_ann = {
            'info': {'description': 'Object Detection Dataset created with Florence-2', 'version': '1.0', 'year': 2025},
            'licenses': [],
            'images': [{'id': image_id, 'file_name': image_filename, 'width': width, 'height': height}],
            'annotations': [],
            'categories': [{'id': 1, 'name': 'person'}, {'id': 2, 'name': 'pet'}]
        }

        for ann_id, ann in enumerate(annotations):
            x1, y1, x2, y2 = ann['bbox']
            category_id = 1 if ann['class'] == 'person' else 2
            coco_ann['annotations'].append({
                'id': ann_id,
                'image_id': image_id,
                'category_id': category_id,
                'bbox': [x1, y1, x2 - x1, y2 - y1],
                'area': ann['area'],
                'iscrowd': 0
            })

        return coco_ann


In [ ]:
class DatasetCreator(AnnotationHandler):
    def __init__(self, config):
        super().__init__(config)
        self.setup_directories()
        self.load_florence_model()
        self.load_local_images()

    def create_dataset(self):
        print("Starting relaxed dataset creation with COCO annotations...")

        stats = defaultdict(lambda: {'train': 0, 'val': 0})
        pet_subclass_stats = {pt: {'train': 0, 'val': 0} for pt in self.config.PET_CLASSES}
        all_annotations = {'train': [], 'val': []}
        image_id = 0

        max_total_per_class = 1500
        max_val_per_class = 150  # reasonable fixed split

        for img_path in tqdm(self.image_paths, desc="Processing images"):
            if all(stats[c]['train'] + stats[c]['val'] >= max_total_per_class for c in self.config.TARGET_CLASSES):
                print("\n✅ Dataset requirements met for all classes!")
                break

            try:
                img = Image.open(img_path)
                img = self.resize_image_if_needed(img)
                florence_output = self.run_florence_detection(img)
                annotations = self.filter_and_convert_annotations(florence_output, img.width, img.height)

                if not annotations:
                    continue

                # Count class presence
                person_present = any(ann['class'] == 'person' for ann in annotations)
                pet_present = any(ann['class'] == 'pet' for ann in annotations)

                # Skip if no relevant classes
                if not (person_present or pet_present):
                    continue

                # Skip overcrowded or noisy
                person_count = sum(1 for ann in annotations if ann['class'] == 'person')
                pet_labels = [ann['original_label'].lower() for ann in annotations if ann['class'] == 'pet']
                if person_count > 4 or any('crowd' in label for label in pet_labels):
                    continue

                split = None
                target_class = None

                # Prefer validation if under quota
                if pet_present and stats['pet']['val'] < max_val_per_class:
                    split = 'val'
                    target_class = 'pet'
                elif pet_present and stats['pet']['train'] + stats['pet']['val'] < max_total_per_class:
                    split = 'train'
                    target_class = 'pet'
                elif person_present and stats['person']['val'] < max_val_per_class:
                    split = 'val'
                    target_class = 'person'
                elif person_present and stats['person']['train'] + stats['person']['val'] < max_total_per_class:
                    split = 'train'
                    target_class = 'person'

                if not split or not target_class:
                    continue

                # Save image with unique name
                image_filename = f"img_{image_id:06d}.jpg"
                split_dir = self.config.VAL_DIR if split == 'val' else self.config.TRAIN_DIR
                final_path = os.path.join(split_dir, "images", image_filename)
                img.save(final_path, 'JPEG', quality=95)

                # Save annotation
                coco_ann = self.create_coco_format_annotation(
                    image_id, image_filename, img.width, img.height, annotations
                )
                all_annotations[split].append(coco_ann)

                stats[target_class][split] += 1

                # Track pet subclasses
                if target_class == 'pet':
                    for ann in annotations:
                        if ann['class'] == 'pet':
                            pet_type = self.get_pet_type(ann['original_label'].lower())
                            if pet_type in pet_subclass_stats:
                                pet_subclass_stats[pet_type][split] += 1

                image_id += 1

                if image_id % 50 == 0:
                    print(f"\nProgress:")
                    for cls in self.config.TARGET_CLASSES:
                        print(f"  {cls.capitalize()} - Train: {stats[cls]['train']}, Val: {stats[cls]['val']}")
                    print("  Pet Subclasses:")
                    for pt in self.config.PET_CLASSES:
                        print(f"    {pt.capitalize()} - Train: {pet_subclass_stats[pt]['train']}, Val: {pet_subclass_stats[pt]['val']}")

            except Exception as e:
                print(f"Error processing {img_path}: {e}")
                continue

        # Save final COCO JSONs
        for split in ['train', 'val']:
            ann_file = os.path.join(self.config.ANNOTATIONS_DIR, f"{split}_annotations.json")
            merged = {
                'info': {
                    'description': 'Object Detection Dataset created with Florence-2 from Flickr30k',
                    'version': '1.0',
                    'year': 2025
                },
                'licenses': [],
                'images': [],
                'annotations': [],
                'categories': [
                    {'id': 1, 'name': 'person'},
                    {'id': 2, 'name': 'pet'}
                ]
            }

            ann_id = 0
            for ann_data in all_annotations[split]:
                merged['images'].extend(ann_data['images'])
                for ann in ann_data['annotations']:
                    ann['id'] = ann_id
                    merged['annotations'].append(ann)
                    ann_id += 1

            with open(ann_file, 'w') as f:
                json.dump(merged, f, indent=2)

        # Final summary
        print("\n" + "=" * 50)
        print("✅ Dataset creation completed!")
        print("=" * 50)

        print("\nDataset Statistics:")
        for cls in self.config.TARGET_CLASSES:
            print(f"\n{cls.capitalize()}:")
            print(f"  Train: {stats[cls]['train']} images")
            print(f"  Val: {stats[cls]['val']} images")

        print("\nPet Subclass Breakdown:")
        for pt in self.config.PET_CLASSES:
            total = pet_subclass_stats[pt]['train'] + pet_subclass_stats[pt]['val']
            print(f"  {pt.capitalize()} - Train: {pet_subclass_stats[pt]['train']}, Val: {pet_subclass_stats[pt]['val']} (Total: {total})")

        print(f"\nTotal images saved: {image_id}")
        print(f"Images saved to: {self.config.OUTPUT_DIR}")


In [24]:
# TTA and Ensemble explanation functions
def explain_tta_for_annotation_checking():
    """
    Explain how Test Time Augmentation can be used to check annotation correctness
    """
    explanation = """
    Test Time Augmentation (TTA) for Annotation Quality Check:
    
    1. Apply multiple augmentations to each training image:
       - Horizontal flip
       - Rotation (small angles: -10 to +10 degrees)
       - Scale variations (0.8x to 1.2x)
       - Slight color jittering
    
    2. Run Florence-2 on each augmented version
    
    3. Compare detections across augmentations:
       - Consistent detections across augmentations indicate reliable annotations
       - Objects detected in original but not in augmented versions might be edge cases
       - Calculate IoU between corresponding boxes across augmentations
       - Flag annotations with low consistency (IoU < 0.5) for manual review
    
    4. Quality metrics:
       - Consistency score = average IoU across all augmentation pairs
       - Confidence score = percentage of augmentations detecting the object
       - Annotations with consistency < 0.7 or confidence < 0.8 should be reviewed
    
    Implementation approach:
    ```python
    def validate_with_tta(image, annotations):
        augmentations = [
            A.HorizontalFlip(p=1),
            A.Rotate(limit=10, p=1),
            A.RandomScale(scale_limit=0.2, p=1),
            A.ColorJitter(brightness=0.2, contrast=0.2, p=1)
        ]
        
        aug_detections = []
        for aug in augmentations:
            aug_image = aug(image=image)['image']
            detections = run_florence_detection(aug_image)
            aug_detections.append(detections)
        
        # Compare detections and calculate consistency
        consistency_scores = calculate_consistency(annotations, aug_detections)
        return consistency_scores
    ```
    """
    return explanation

def explain_ensemble_for_annotation_checking():
    """
    Explain how ensembles can be used to check annotation correctness
    """
    explanation = """
    Ensemble Methods for Annotation Quality Check:
    
    1. Use multiple pre-trained models:
       - Florence-2-large (primary)
       - Florence-2-base (secondary)
       - Other vision-language models like OWL-ViT or GLIP
    
    2. Ensemble voting strategy:
       - Run all models on the same image
       - Match detections across models using IoU threshold (0.5)
       - Apply majority voting: keep annotations detected by >50% of models
    
    3. Confidence scoring:
       - High confidence: All models agree (IoU > 0.7)
       - Medium confidence: Majority models agree (IoU > 0.5)
       - Low confidence: Only one model detects
    
    4. Quality control:
       - Automatically accept high confidence annotations
       - Flag medium confidence for quick review
       - Manually review or discard low confidence annotations
    
    5. Benefits:
       - Reduces model-specific biases
       - Improves annotation reliability
       - Identifies ambiguous cases that need human review
    
    Implementation approach:
    ```python
    def validate_with_ensemble(image):
        models = [
            load_florence2_large(),
            load_florence2_base(),
            load_owl_vit()  # if available
        ]
        
        all_detections = []
        for model in models:
            detections = model.detect(image)
            all_detections.append(detections)
        
        # Apply voting mechanism
        final_detections = ensemble_vote(all_detections, iou_threshold=0.5)
        confidence_scores = calculate_confidence(final_detections, all_detections)
        
        return final_detections, confidence_scores
    ```
    """
    return explanation

In [29]:
# Main execution
if __name__ == "__main__":
    config = Config()
    
    # Check if Flickr30k folder exists
    if not os.path.exists(config.FLICKR30K_PATH):
        print(f"ERROR: Flickr30k folder not found at '{config.FLICKR30K_PATH}'")
        print("Please ensure the folder path is correct in the Config class.")
        exit(1)
    
    # Create dataset
    creator = DatasetCreator(config)
    creator.create_dataset()
    
    # Save explanations
    os.makedirs("documentation", exist_ok=True)
    
    with open(os.path.join("documentation", "tta_explanation.txt"), 'w') as f:
        f.write(explain_tta_for_annotation_checking())
    
    with open(os.path.join("documentation", "ensemble_explanation.txt"), 'w') as f:
        f.write(explain_ensemble_for_annotation_checking())
    
    print("\nExplanations for TTA and Ensemble methods saved in documentation folder!")
    
    # Explanation of degrees of freedom decisions
    dof_explanation = """
    Degrees of Freedom Decisions:
    
    1. Object Size (32-640 pixels):
       - Min 32px: Smaller objects are often too ambiguous for reliable detection
       - Max 640px: Larger objects might dominate the image, reducing diversity
       - This range captures most real-world object scales
    
    2. Image Size (224-1024 pixels):
       - Min 224px: Matches common CNN input sizes (ResNet, EfficientNet)
       - Max 1024px: Balances detail preservation with computational efficiency
       - Allows models to learn multi-scale features effectively
    
    3. These constraints ensure:
       - Sufficient object detail for accurate detection
       - Reasonable training time and memory usage
       - Good generalization to real-world scenarios
       - Compatibility with common detection architectures
    
    4. Flickr30k Dataset Benefits:
       - High-quality images with diverse scenes
       - Natural distribution of object sizes and contexts
       - Real-world scenarios that match deployment conditions
    """
    
    with open(os.path.join("documentation", "degrees_of_freedom_explanation.txt"), 'w') as f:
        f.write(dof_explanation)
    
    print("Degrees of freedom explanation saved!")

Loading Florence-2 model...
Model loaded successfully on cuda!
Loading images from flickr30k_images...
Found 6908 images.
Starting relaxed dataset creation with COCO annotations...


Processing images:   2%|▏         | 127/6908 [04:24<3:46:04,  2.00s/it]


Progress:
  Person - Train: 0, Val: 28
  Pet - Train: 0, Val: 22
  Pet Subclasses:
    Dog - Train: 0, Val: 10
    Cat - Train: 0, Val: 1
    Horse - Train: 0, Val: 17


Processing images:   4%|▎         | 251/6908 [09:05<3:27:32,  1.87s/it]


Progress:
  Person - Train: 0, Val: 57
  Pet - Train: 0, Val: 43
  Pet Subclasses:
    Dog - Train: 0, Val: 23
    Cat - Train: 0, Val: 6
    Horse - Train: 0, Val: 25


Processing images:   6%|▌         | 382/6908 [14:40<2:00:27,  1.11s/it] 

... (97 outputs left out)

Processing images:  97%|█████████▋| 6723/6908 [4:07:59<05:29,  1.78s/it]


Progress:
  Person - Train: 1282, Val: 200
  Pet - Train: 918, Val: 200
  Pet Subclasses:
    Dog - Train: 280, Val: 70
    Cat - Train: 333, Val: 69
    Horse - Train: 657, Val: 130


Processing images: 100%|██████████| 6908/6908 [4:13:35<00:00,  2.20s/it]


✅ Dataset creation completed!

Dataset Statistics:

Person:
  Train: 1300 images
  Val: 200 images

Pet:
  Train: 945 images
  Val: 200 images

Pet Subclass Breakdown:
  Dog - Train: 290, Val: 70 (Total: 360)
  Cat - Train: 342, Val: 69 (Total: 411)
  Horse - Train: 673, Val: 130 (Total: 803)

Total images saved: 2645
Images saved to: object_detection_dataset

Explanations for TTA and Ensemble methods saved in documentation folder!
Degrees of freedom explanation saved!
